# Auditoria da necessidade de retreinamento por genoma

Este notebook lê os **splits reais de fragmentos curtos**. Uma inspeção curta mostra comprimentos e alfabetos antes da varredura completa; linhas fora do protocolo de até 512 bp são contabilizadas separadamente, sem interromper a auditoria. Indexa **todo o treino** em SQLite temporário, mede correspondências exatas entre teste e treino (incluindo fita reversa complementar), após remover o padding `_` à direita como no notebook histórico de benchmark e verifica se o teste possui genomas de origem ausentes do treino. Não carrega checkpoints, não treina modelos e não cria dados sintéticos.

**Interpretação:** retreinar só é necessário para responder à pergunta de generalização para *genomas não vistos* quando o checkpoint existente já foi treinado nos genomas disponíveis para teste. Para relatar o experimento histórico no seu escopo de referências conhecidas, é possível manter o checkpoint, mas a sobreposição precisa ser declarada e as alegações limitadas. Um teste sem duplicatas exatas não prova independência genômica.

Se `genome_id` ou `accession` não está associado a cada **fragmento** nos dois splits, o resultado será `PROVENANCE_REQUIRED`: uma lista de genomas obtida posteriormente pelo nome da espécie não estabelece a origem do fragmento.

**Custo:** o modo completo lê aproximadamente 15 milhões de fragmentos de treino e 1,7 milhão de teste; reserve horas e espaço em disco para o índice temporário. `MAX_TEST_ROWS` serve só para verificar o funcionamento inicial e nunca produz uma decisão final.

**Atenção:** uma execução anterior a esta correção pode subestimar correspondências reverso complementares em sequências preenchidas com `_`. Reexecute a auditoria completa e compare o campo `sequence_normalization` no novo `decision.json`. A taxa antiga de 79,98% não é a estimativa final para esse protocolo.

## Entradas e namespace M01

Configure `M01_CONFIG` com um YAML (modelo em `configs/reproduction_m01.yaml`) ou `M01_RUN_PLAN` com o plano M00 selecionado. O plano fixa entradas e destino; overrides conflitantes são recusados. Sem plano, cada execução reserva novo `run_id`. `DATA_ROOT`, `ORIGINAL_MAPPING_PATH`, `TAXONOMY_PATH` e os overrides descritos em `docs/m01_data_taxonomy_split.md` são aceitos sem plano. Saídas históricas permanecem somente leitura.

O modo padrão é empírico, inclusive se faltarem dados. Testes locais usam `REMOTE_SERVER=False SMOKE_TEST=1` com fixtures explícitas. Manifesto COMPLETE significa análise concluída, não integridade taxonômica nem independência genômica. Examine o relatório e `hierarchy_ready` antes de usar a hierarquia.


In [ ]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        PROJECT_ROOT = r
        break
else:
    raise FileNotFoundError("Execute dentro do repositório cnn_metagenome")

from viral_cnn.m01_workflow import run_m01_notebook


In [ ]:
manifest = run_m01_notebook("short/10", PROJECT_ROOT)
print("Run:", manifest["run_id"])
print("Saída:", manifest["output_dir"])
print("Status:", manifest["status"])


## Evidência e próximo passo

O manifesto registra caminhos, hashes ou identidade limitada explícita, configuração, versões e origem sintética/observada. Consulte `output_dir` abaixo; não procure a execução mais recente por data. A taxa de overlap usa o teste integral como denominador; piloto não estima prevalência. A ausência de duplicata exata não prova independência genômica. Estes resultados não são o benchmark C03.


In [ ]:
import json
print(json.dumps(manifest["metrics_summary"], indent=2, ensure_ascii=False))
